# STDP 学习规则更新权重

In [1]:
from network import *
from optimizer import STDP
from functools import partial
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [2]:
def i_fn(t, freq, duration, amplitude):
    if t % freq < duration:
        return amplitude
    return 0

current_fn1 = partial(i_fn, freq=10, duration=5, amplitude=20)
current_fn2 = partial(i_fn, freq=12, duration=5, amplitude=20)
current_fn3 = partial(i_fn, freq=14, duration=5, amplitude=20)
current_fn4 = partial(i_fn, freq=16, duration=5, amplitude=20)

In [3]:
data = load_iris()
x, y = data.data, data.target
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
scaler = MinMaxScaler()
scaler.fit(x_train)
x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)

In [4]:
T = 500
dt = 0.01
n_t = int(T / dt)
time = np.arange(0, T + dt, dt)

network = Network([
    NeuronGroup(4, l=1000, d=10.0, r_a=25.0),
    NeuronGroup(32, l=1000, d=10.0, r_a=25.0),
], g=0.4, e_syn=-65.0, delay=0.5)
network.add_current_injector(current_fn1, connect_pattern=lambda i, j: j == 0, delay=0.5)
network.add_current_injector(current_fn2, connect_pattern=lambda i, j: j == 1, delay=0.5)
network.add_current_injector(current_fn3, connect_pattern=lambda i, j: j == 2, delay=0.5)
network.add_current_injector(current_fn4, connect_pattern=lambda i, j: j == 3, delay=0.5)

optimizer = STDP(network.synapses, f_pos=lambda x: 0.01 * (1 if x <= 1.5 else 0), f_neg=lambda x: 0.01 * (1 if x >= -1.5 else 0))

In [5]:
epochs = 4
weights1 = [network.synapses[0].synapses[0].weight]
weights2 = [network.synapses[0].synapses[32].weight]
weights3 = [network.synapses[0].synapses[64].weight]
weights4 = [network.synapses[0].synapses[96].weight]
for i in range(epochs):
    for j, x in tqdm(enumerate(x_train)):
        current_fn1 = partial(i_fn, freq=(-x[0] + 2) * 8, duration=5, amplitude=20)
        current_fn2 = partial(i_fn, freq=(-x[1] + 2) * 8, duration=5, amplitude=20)
        current_fn3 = partial(i_fn, freq=(-x[2] + 2) * 8, duration=5, amplitude=20)
        current_fn4 = partial(i_fn, freq=(-x[3] + 2) * 8, duration=5, amplitude=20)
        network.injectors[0].i_fn = current_fn1
        network.injectors[1].i_fn = current_fn2
        network.injectors[2].i_fn = current_fn3
        network.injectors[3].i_fn = current_fn4

        network.update_recoder(T)
        for t in range(n_t):
            network.step(dt)
            optimizer.step(dt)

        optimizer.apply()
        weights1.append(network.synapses[0].synapses[0].weight)
        weights2.append(network.synapses[0].synapses[32].weight)
        weights3.append(network.synapses[0].synapses[64].weight)
        weights4.append(network.synapses[0].synapses[96].weight)
        network.reset()

120it [2:54:39, 87.33s/it]
120it [2:54:50, 87.42s/it]
120it [2:55:01, 87.51s/it]
120it [2:55:02, 87.52s/it]
120it [2:55:06, 87.55s/it]


In [33]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(weights1)), y=weights1, name="Synapse 1"))
fig.add_trace(go.Scatter(x=np.arange(len(weights2)), y=weights2, name="Synapse 2"))
fig.add_trace(go.Scatter(x=np.arange(len(weights3)), y=weights3, name="Synapse 3"))
fig.add_trace(go.Scatter(x=np.arange(len(weights4)), y=weights4, name="Synapse 4"))
fig.update_xaxes(range=[0, 120])
fig.update_yaxes(range=[-1.5, 1.5])
fig.update_layout(title="STDP Learning Rule", xaxis_title="Iters", yaxis_title="Weight")
fig.show()